# Stage 04 — Hybrid retrieval

**Track A (Buse) · Stage 4 of 10**

| | |
|---|---|
| **Input** | The Chroma collection, plus `data/processed/chunks.jsonl` for the sparse index |
| **Output** | A `search(query) -> candidates` function, and the top-k contract Sude's tools call |
| **Promotes to** | `bm25_B.py`, `hybrid_B.py`, `service_B.py` in `src/research_assistant/retrieval/` |
| **Config** | `configs/retrieval_B.yaml` |

## Why hybrid rather than pure vector

The two arms fail differently, which is the whole argument. Dense retrieval misses
exact identifiers: a model name, a dataset name, a metric abbreviation. Sparse
retrieval misses paraphrase. Academic questions contain both, constantly.

## Design choices

| Choice | Picked | Alternatives | Why |
|---|---|---|---|
| Sparse arm | BM25 in memory | Elasticsearch, SPLADE | At corpus scale here, a pickled index is the whole story. SPLADE is a learned sparse model worth a ledger entry, not a dependency. |
| Fusion | Reciprocal rank fusion | Weighted score sum | Rank-based fusion needs no score calibration between two arms whose scores are on incomparable scales. Weighted sum needs normalisation, and the normalisation becomes another thing to tune and defend. |
| Candidates per arm | 30 | 10, 50 | Enough that fusion has room to disagree. |
| Candidate set | 20 | 10, 50 | The number handed to the reranker. It caps recall permanently: whatever misses here is unrecoverable in stage 08, which is why `recall_at_20` is a gate threshold. |
| Final top-k | 5 | 3, 10 | What the tools return. More context is not free, it dilutes the writer agent and raises the chance of an unfaithful citation. |

**The most useful experiment here** is the ablation: BM25 alone, vector alone, fused.
It gives you a table for the report and it tells you whether hybrid is earning its
complexity on your corpus, rather than because it is conventional.

In [ ]:
from _nbsetup_B import REPO, load_cfg, resolve
import json, pickle
from pathlib import Path

rcfg = load_cfg("retrieval")
icfg = load_cfg("ingestion")
chunks = [json.loads(l) for l in (resolve(icfg["corpus"]["processed_dir"]) / "chunks.jsonl")
          .read_text(encoding="utf-8").splitlines() if l.strip()]
by_id = {c["chunk_id"]: c for c in chunks}
print(len(chunks), "chunks")

In [ ]:
import re
from rank_bm25 import BM25Okapi

b = rcfg["bm25"]
STOP = set("""a an the of and or to in for on with is are was were be been this that
these those we our it its as by from at can may should not no than then so such""".split())

def tokenize(s):
    toks = re.findall(r"[a-z0-9]+", s.lower())
    return [t for t in toks if t not in STOP] if b["stopwords"] == "english" else toks

corpus_ids = [c["chunk_id"] for c in chunks]
bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks], k1=b["k1"], b=b["b"])

def bm25_search(query, n):
    scores = bm25.get_scores(tokenize(query))
    ranked = sorted(zip(corpus_ids, scores), key=lambda x: -x[1])[:n]
    return [cid for cid, _ in ranked]

with resolve(b["index_path"]).open("wb") as f:
    pickle.dump({"bm25": bm25, "ids": corpus_ids}, f)
print("bm25 index built and pickled")

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

vs, e = rcfg["vector_store"], icfg["embed"]
client = chromadb.PersistentClient(path=str(resolve(vs["persist_dir"])))
coll = client.get_collection(vs["collection"])
model = SentenceTransformer(e["model"])

def vector_search(query, n):
    v = model.encode([e["query_prefix"] + query], normalize_embeddings=e["normalize"])[0]
    res = coll.query(query_embeddings=[v.tolist()], n_results=n)
    return res["ids"][0]

In [ ]:
h = rcfg["hybrid"]

def rrf(rank_lists, k):
    # Reciprocal rank fusion: score = sum over arms of 1 / (k + rank).
    # Rank-based, so the two arms' incomparable score scales never meet.
    scores = {}
    for ids in rank_lists:
        for rank, cid in enumerate(ids, start=1):
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank)
    return [cid for cid, _ in sorted(scores.items(), key=lambda x: -x[1])]

def hybrid_search(query, candidate_k=None):
    n = h["candidates_per_arm"]
    fused = rrf([bm25_search(query, n), vector_search(query, n)], h["rrf_k"])
    return [by_id[cid] for cid in fused[: (candidate_k or h["candidate_k"])]]

q = "TODO: a real question from your domain brief"
for c in hybrid_search(q)[:5]:
    print(f"{c['title'][:45]:45s} | {c['section'][:30]:30s} p{c['page_start']}")

### The ablation

Fill `ndcg5` from notebook 06. This table goes straight into the report, and it is
the evidence that decides whether the sparse arm stays.

In [ ]:
ARMS = {
    "bm25_only":   lambda q, n: bm25_search(q, n),
    "vector_only": lambda q, n: vector_search(q, n),
    "hybrid_rrf":  lambda q, n: rrf([bm25_search(q, n), vector_search(q, n)], h["rrf_k"])[:n],
}

import pandas as pd
pd.DataFrame([dict(arm=name, ndcg5=None, recall20=None, latency_ms=None) for name in ARMS])

## The contract this stage owns

Everything above is yours to change freely except the **shape of what `search` returns**.
That shape is the interface Sude's tools serialise, and it is written down in
`src/research_assistant/contracts/retrieval_J.py`, a joint file. Agree changes with
her before making them.

The other seam: the top-k step must load its ranker from the registry rather than
calling a ranking function directly. That indirection is what lets your tuned model
from stage 08 replace the baseline without touching the tools. Build it now, when it
costs one function, not in stage 09 when it costs a refactor.

## Exit checks

- [ ] `recall@20` on the stage 05 query set clears 0.85. Below that, fix chunking or
      fusion. Reranking cannot recover a chunk that never entered the candidate set.
- [ ] BM25 and vector each win on at least a few queries. If one arm never wins,
      hybrid is paying complexity for nothing and the ledger should record that.
- [ ] Query latency is measured and written down. It is a gate cost, not a footnote.
- [ ] Ranker selection goes through the registry.

## Promote to `src/`

`bm25_B.py` gets the tokenizer and index. `hybrid_B.py` gets fusion. `service_B.py`
becomes the one entry point that loads config, runs both arms, fuses, reranks via the
registry, and returns the contract type.